# Day3_16 · 센서값으로 상태 확률 계산

온도·습도·조도·CO₂ 센서값으로 실내 재실 확률을 계산하고 기준값에 따라 0/1 판단을 만듭니다.

## 이 노트북에서 확인할 내용

- 0/1 결과와 예측확률을 구분할 수 있습니다.
- 학습자료에서 익힌 표준화 규칙을 평가자료에 적용할 수 있습니다.
- 기준값을 바꾸면 최종 판단 수가 달라지는 이유를 설명할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-16-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-16-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-16-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-16-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-16-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-16-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-16-07 · 실내 재실자료 읽기

In [ ]:
occupancy = pd.read_csv(DATA_DIR / 'occupancy_detection.csv')

In [ ]:
occupancy.head()

### D3B-16-08 · 자료 크기 확인

In [ ]:
occupancy.shape

### D3B-16-09 · 열 이름 확인

In [ ]:
occupancy.columns

### D3B-16-10 · 날짜와 시간을 날짜 형식으로 바꾸기

In [ ]:
occupancy['date_time'] = pd.to_datetime(occupancy['date_time'])

In [ ]:
occupancy[['date_time']].head()

### D3B-16-11 · 학습·평가 구분 확인

In [ ]:
occupancy['source_split'].value_counts()

### D3B-16-12 · 빈 방과 재실 행 수 확인

In [ ]:
occupancy['occupied'].value_counts().sort_index()

## 센서값으로 재실 확률 계산하기

결과 열 `occupied`는 빈 방이면 0, 사람이 있으면 1입니다. 로지스틱 회귀는 여러 센서값을 이용해 1일 확률을 계산합니다.

### D3B-16-13 · 입력 센서 열 정하기

In [ ]:
sensor_columns = ['temperature_c', 'humidity_percent', 'light_lux', 'co2_ppm', 'humidity_ratio']

In [ ]:
pd.Series(sensor_columns, name='sensor')

### D3B-16-14 · 학습 행 선택

In [ ]:
occupancy_train = occupancy.loc[occupancy['source_split'] == 'train'].copy()

In [ ]:
occupancy_train.head()

### D3B-16-15 · 평가 행 선택

In [ ]:
occupancy_test = occupancy.loc[occupancy['source_split'] != 'train'].copy()

In [ ]:
occupancy_test.head()

### D3B-16-16 · 학습 입력표 만들기

In [ ]:
X_train = occupancy_train[sensor_columns]

In [ ]:
X_train.head()

### D3B-16-17 · 학습 결과열 만들기

In [ ]:
y_train = occupancy_train['occupied']

In [ ]:
y_train.head()

### D3B-16-18 · 평가 입력표 만들기

In [ ]:
X_test = occupancy_test[sensor_columns]

In [ ]:
X_test.head()

### D3B-16-19 · 평가 결과열 만들기

In [ ]:
y_test = occupancy_test['occupied']

In [ ]:
y_test.head()

센서마다 숫자 단위가 크게 다릅니다. 학습자료에서 각 열의 평균과 표준편차를 구해 비슷한 숫자 범위로 바꿉니다.

### D3B-16-20 · 표준화 도구 불러오기

In [ ]:
from sklearn.preprocessing import StandardScaler

### D3B-16-21 · 학습자료에서 변환 규칙 익히기

In [ ]:
scaler = StandardScaler().fit(X_train)

In [ ]:
scaler

### D3B-16-22 · 학습 입력값 변환하기

In [ ]:
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=sensor_columns, index=X_train.index)

In [ ]:
X_train_scaled.head()

### D3B-16-23 · 평가 입력값 변환하기

In [ ]:
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=sensor_columns, index=X_test.index)

In [ ]:
X_test_scaled.head()

### D3B-16-24 · 변환 전후 한 행 비교

In [ ]:
scaling_example = pd.DataFrame({'before': X_train.iloc[0], 'after': X_train_scaled.iloc[0]})

In [ ]:
scaling_example.round(3)

### D3B-16-25 · 로지스틱 회귀 불러오기

In [ ]:
from sklearn.linear_model import LogisticRegression

### D3B-16-26 · 재실 확률모형 학습하기

In [ ]:
occupancy_model = LogisticRegression(max_iter=2000, random_state=42).fit(X_train_scaled, y_train)

In [ ]:
occupancy_model

### D3B-16-27 · 평가행의 재실 확률 계산

In [ ]:
occupied_probability = occupancy_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
pd.Series(occupied_probability[:10], name='probability_occupied')

### D3B-16-28 · 실제 상태와 확률을 한 표로 만들기

In [ ]:
probability_table = occupancy_test[['date_time', 'occupied']].copy()
probability_table['probability_occupied'] = occupied_probability

In [ ]:
probability_table.head(10).round(3)

확률은 0과 1 사이의 연속된 값입니다. 최종 0/1 판단을 만들려면 어느 확률부터 재실로 볼지 기준값을 정해야 합니다.

### D3B-16-29 · 기본 기준값 정하기

In [ ]:
decision_threshold = 0.5

In [ ]:
decision_threshold

### D3B-16-30 · 확률을 0과 1로 바꾸기

In [ ]:
occupancy_prediction = (occupied_probability >= decision_threshold).astype(int)

In [ ]:
pd.Series(occupancy_prediction[:10], name='predicted_occupied')

### D3B-16-31 · 확률표에 최종 판단 추가

In [ ]:
probability_table['predicted_occupied'] = occupancy_prediction

In [ ]:
probability_table.head(15).round(3)

### D3B-16-32 · 실제 상태별 확률 분포 그리기

In [ ]:
sns.histplot(data=probability_table, x='probability_occupied', hue='occupied', bins=30, element='step', stat='density', common_norm=False)
plt.axvline(decision_threshold, color='black', linestyle='--')
plt.show()

### D3B-16-33 · 센서별 회귀계수 표 만들기

In [ ]:
logistic_coefficients = pd.DataFrame({'sensor': sensor_columns, 'coefficient': occupancy_model.coef_[0]}).sort_values('coefficient')

In [ ]:
logistic_coefficients.round(3)

### D3B-16-34 · 회귀계수 막대그래프

In [ ]:
logistic_coefficients.plot.barh(x='sensor', y='coefficient', legend=False)
plt.xlabel('Coefficient after scaling')
plt.show()

양의 계수는 다른 센서값이 같을 때 해당 센서값이 커질수록 재실 확률이 높아지는 방향을 나타냅니다. 센서 간 상관과 시간 구조가 있으므로 계수를 센서의 인과효과로 해석하지 않습니다.

### D3B-16-35 · 바꾸어 볼 판단 기준 정하기

In [ ]:
# 0.5를 0.3 또는 0.7로 바꾸어 보세요.
chosen_threshold = 0.3

In [ ]:
chosen_threshold

### D3B-16-36 · 새 기준으로 최종 판단 만들기

In [ ]:
chosen_prediction = (occupied_probability >= chosen_threshold).astype(int)

In [ ]:
pd.Series(chosen_prediction).value_counts().sort_index()

### D3B-16-37 · 기준값별 재실 판단 수 비교

In [ ]:
threshold_count_table = pd.DataFrame({
    'threshold_0.5': pd.Series(occupancy_prediction).value_counts(),
    f'threshold_{chosen_threshold}': pd.Series(chosen_prediction).value_counts(),
}).fillna(0).astype(int)

In [ ]:
threshold_count_table

### D3B-16-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''probability_table을 사용하세요. 기존 셀은 수정하지 마세요. 재실 확률을 시간 순서로 보여 주고 0.5 기준선을 표시하는 그래프 한 개를 제안해 주세요. 확률과 최종 0/1 판단이 다른 결과라는 점을 두 문장으로 설명해 주세요.'''.strip()

In [ ]:
codex_request

### D3B-16-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['시간 순서가 유지되는가?', '확률축이 0에서 1 사이인가?', '기준값이 모델 자체와 같다고 설명하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist